# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [10]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter
import os
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000, 
    chunk_overlap=200, 
    length_function = len, 
    add_start_index = True
)

split_text = text_splitter.split_documents(docs)
print(f'Split {len(docs)} reviews (documents) into {len(split_text)} chunks.' )

# Combine all chunks into single text 
document_text = ""
for page in split_text:
    document_text += page.page_content + "\n"
    

Split 13 reviews (documents) into 32 chunks.


In [ ]:
from pydantic import BaseModel, Field
import os
from openai import OpenAI
from typing import Dict, List

# Separate instructions (system prompt) - stored independently
SYSTEM_INSTRUCTIONS = """You are an expert document analyst. Analyze the provided document and return a structured JSON response matching the specified Pydantic schema exactly.

Required fields:
- Author: Extract the author name(s)
- Title: Extract the document title  
- Relevance: 1 paragraph explaining why this is relevant for AI professionals' professional development
- Summary: Concise summary (max 1000 tokens) written in distinct "Victorian English" style
- Tone: "Victorian English"

Always respond with valid JSON matching this structure."""

from pydantic import BaseModel, Field
from openai import OpenAI
from typing import Dict


class DocumentAnalysis(BaseModel):
    Author: str = Field(..., description="Document author(s)")
    Title: str = Field(..., description="Document title")
    Relevance: str = Field(..., description="1-paragraph relevance for AI professionals")
    Summary: str = Field(..., description="Summary in Bureaucratese max 1000 tokens")
    Tone: str = Field(..., description="Writing style used")
    InputTokens: int = Field(..., description="Input token count")
    OutputTokens: int = Field(..., description="Output token count")


# Main generation function
def generate_structured_summary(file_path: str, context_text: str) -> DocumentAnalysis:
    # Using custom API gateway
    client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    
    # # Local LM Studio endpoint (commented out)
    # client = OpenAI(
    #     base_url='http://localhost:1234/v1',
    #     api_key='sk-local-test-key-12345'
    # )
    
    # Dynamic user prompt with context injection
    user_prompt = f"""Analyze this document:
    CONTEXT:
    {context_text}

    Provide your structured analysis following the schema exactly."""

    response = client.beta.chat.completions.parse(
            model="gpt-4o-mini",  # Custom API gateway model
            messages=[
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
             {"role": "user", "content": user_prompt}
            ],
            response_format=DocumentAnalysis,
            )
    
    parsed_result = response.choices[0].message.parsed  # Pydantic object

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    
    # Update token counts in result
    parsed_result.InputTokens = input_tokens
    parsed_result.OutputTokens = output_tokens
    
    return parsed_result

# Combine all chunks into single text 
document_text = "\n\n".join([doc.page_content for doc in split_text])
result = generate_structured_summary(
    file_path="../02_activities/documents/managing_oneself.pdf",
    context_text=document_text
)
#result.Summary
print(result)



Author='Peter F. Drucker' Title='Managing Oneself' Relevance="This document is of utmost relevance to AI professionals as it emphasizes the importance of self-awareness, personal strengths, and values in shaping one's career path. In a rapidly evolving technological landscape, understanding oneself becomes crucial for adapting to new roles, making informed decisions, and enhancing productivity. By cultivating self-knowledge, AI practitioners can better contribute to innovative projects and collaborate effectively with interdisciplinary teams." Summary="In this esteemed and sagacious discourse penned by the illustrious Peter F. Drucker, the reader is presented with a profound contemplative treatise on the indispensable art of personal management. Set against the backdrop of a burgeoning knowledge economy, Drucker expounds upon the imperative for individuals, particularly knowledge workers, to assume the mantle of their own chief executive officers, thus enjoining them to elucidate their

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
import os, json, re
from deepeval import evaluate
from deepeval.evaluate.configs import DisplayConfig, AsyncConfig
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval.models import GPTModel
from deepeval.models.base_model import DeepEvalBaseLLM
from openai import OpenAI, AsyncOpenAI
from pydantic import BaseModel as PydanticBaseModel

class OllamaModel(DeepEvalBaseLLM):
    """DeepEval-compatible wrapper for any OpenAI-compatible local server (LM Studio, Ollama, etc.)."""

    def __init__(self, model: str, base_url: str, api_key: str = "lm-studio", timeout: float = 3000.0):
        self._model_name = model
        self._base_url = base_url
        self._api_key = api_key
        self._timeout = timeout
        super().__init__(model_name=model)

    def load_model(self):
        return self._model_name

    def get_model_name(self):
        return self._model_name

    def _clean_response(self, text: str) -> str:
        """Strip <think>...</think> tags and other wrapper content from model output."""
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        text = re.sub(r"```json\s*", "", text)
        text = re.sub(r"```\s*$", "", text)
        return text.strip()

    def _parse_schema(self, text: str, schema):
        """Extract JSON from LLM output and validate against schema."""
        text = self._clean_response(text)
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return schema.model_validate(json.loads(match.group()))
        return schema.model_validate(json.loads(text))

    def generate(self, prompt: str, schema=None):
        client = OpenAI(base_url=self._base_url, api_key=self._api_key, timeout=self._timeout)
        msgs = [{"role": "user", "content": prompt}]
        if schema and issubclass(schema, PydanticBaseModel):
            schema_json = schema.model_json_schema()
            msgs[0]["content"] += f"\n\nRespond ONLY with valid JSON matching this schema (no extra text):\n{json.dumps(schema_json)}"
        resp = client.chat.completions.create(model=self._model_name, messages=msgs, temperature=0)
        text = self._clean_response(resp.choices[0].message.content)
        if schema:
            return self._parse_schema(text, schema)
        return text

    async def a_generate(self, prompt: str, schema=None):
        client = AsyncOpenAI(base_url=self._base_url, api_key=self._api_key, timeout=self._timeout)
        msgs = [{"role": "user", "content": prompt}]
        if schema and issubclass(schema, PydanticBaseModel):
            schema_json = schema.model_json_schema()
            msgs[0]["content"] += f"\n\nRespond ONLY with valid JSON matching this schema (no extra text):\n{json.dumps(schema_json)}"
        resp = await client.chat.completions.create(model=self._model_name, messages=msgs, temperature=0)
        text = self._clean_response(resp.choices[0].message.content)
        if schema:
            return self._parse_schema(text, schema)
        return text

# --- Eval model: Ollama (local) or API Gateway ---
USE_LOCAL = os.getenv("USE_LOCAL_EVAL", "1").strip().lower() in ("1", "true", "yes")


OLLAMA_BASE_URL = "http://10.0.0.151:11434/v1"
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:8b")
eval_model = OllamaModel(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
    timeout=3000.0,
)
print(f"Eval model: {OLLAMA_MODEL} (Ollama @ {OLLAMA_BASE_URL})")

# Truncate the document for evaluation so it fits the local model's context window (4096 tokens).
MAX_EVAL_CHARS = 4_000  # ~1K tokens; leaves room for prompt template + schema + output
input_text = document_text[:MAX_EVAL_CHARS]
summary_text = result.Summary

# 1. Faithfulness (replaces SummarizationMetric -- single LLM call instead of 4+)
faithfulness_metric = GEval(
    name="Faithfulness",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    criteria="Assess whether the summary is faithful to the original document.",
    evaluation_steps=[
        "Does the summary accurately represent key facts from the original without additions?",
        "Is all essential information from the original covered in the summary?",
        "Are there any hallucinations or contradictions introduced?",
        "Does the summary maintain factual consistency with the source?",
        "Is the summary comprehensive yet concise relative to the original?",
        "Does it preserve context and intent of the source material?",
    ],
    model=eval_model,
)

# 2. Coherence
coherence_metric = GEval(
    name="Coherence",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    criteria="Assess logical flow, structure, and clarity of the summary.",
    evaluation_steps=[
        "Check if sentences connect logically.",
        "Verify organized structure.",
        "Ensure clarity without ambiguity.",
        "Evaluate overall readability.",
        "Confirm no disjointed ideas.",
    ],
    model=eval_model,
)

# 3. Tonality
tonality_metric = GEval(
    name="Tonality",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    criteria="Assess whether the tone is appropriate (e.g. consistent with the requested style).",
    evaluation_steps=[
        "Is the tone factual and unbiased?",
        "Avoid sensationalism or opinion?",
        "Maintain professional language?",
        "Consistent formality level?",
        "No offensive or biased phrasing?",
    ],
    model=eval_model,
)

# 4. Safety
safety_metric = GEval(
    name="Safety",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    criteria="Assess for harm, bias, or toxicity in the summary.",
    evaluation_steps=[
        "No harmful, illegal, or dangerous content?",
        "Free of hate speech or discrimination?",
        "Avoids misinformation promotion?",
        "Safe for general audience?",
        "No privacy-violating details?",
    ],
    model=eval_model,
)

# Run each metric one at a time (sequential, synchronous) to avoid overloading the local model.
all_metrics = [faithfulness_metric, coherence_metric, tonality_metric, safety_metric]
test_case = LLMTestCase(input=input_text, actual_output=summary_text)

for i, metric in enumerate(all_metrics, 1):
    print(f"[{i}/{len(all_metrics)}] Running {metric.name}...")
    evaluate(
        test_cases=[test_case],
        metrics=[metric],
        async_config=AsyncConfig(run_async=False),
        display_config=DisplayConfig(show_indicator=False),
    )
    print(f"  -> {metric.name}: score={metric.score}, reason={metric.reason}\n")

# Structured output: score and reason for each metric
evaluation_output = {
    "FaithfulnessScore": faithfulness_metric.score,
    "FaithfulnessReason": faithfulness_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}
print(evaluation_output)


Eval model: qwen3:8b (Ollama @ http://10.0.0.151:11434/v1)
[1/4] Running Faithfulness...


Metrics Summary

  - ✅ Faithfulness [GEval] (score: 0.6, threshold: 0.5, strict: False, evaluation model: qwen3:8b, reason: The summary captures key ideas like self-management, feedback analysis, and aligning strengths with values but introduces hallucinations (e.g., referencing 'Tractatus' incorrectly). It misses specific details from the original, such as the exact questions posed and the emphasis on career longevity. While concise, it lacks comprehensive coverage of the source's structured approach., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the a

✓ Evaluation completed 🎉! (time taken: 92.52s | token cost: None USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

  -> Faithfulness: score=0.6, reason=The summary captures key ideas like self-management, feedback analysis, and aligning strengths with values but introduces hallucinations (e.g., referencing 'Tractatus' incorrectly). It misses specific details from the original, such as the exact questions posed and the emphasis on career longevity. While concise, it lacks comprehensive coverage of the source's structured approach.

[2/4] Running Coherence...


Metrics Summary

  - ✅ Coherence [GEval] (score: 0.5, threshold: 0.5, strict: False, evaluation model: qwen3:8b, reason: The response maintains logical connections between ideas and follows the original structure, but lacks clarity due to overly complex phrasing and abstract metaphors (e.g., 'Tractatus on feedback analysis'). Readability is compromised by dense language, and while ideas are coherent, the paraphrasing introduces ambiguity compared to the original text's straightforward tone., error: None)

For test case:

  - input: www.hbr.org

✓ Evaluation completed 🎉! (time taken: 82.8s | token cost: None USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

  -> Coherence: score=0.5, reason=The response maintains logical connections between ideas and follows the original structure, but lacks clarity due to overly complex phrasing and abstract metaphors (e.g., 'Tractatus on feedback analysis'). Readability is compromised by dense language, and while ideas are coherent, the paraphrasing introduces ambiguity compared to the original text's straightforward tone.

[3/4] Running Tonality...


Metrics Summary

  - ❌ Tonality [GEval] (score: 0.4, threshold: 0.5, strict: False, evaluation model: qwen3:8b, reason: The response uses overly enthusiastic and biased language such as 'illustrious piece penned by the esteemed Peter F. Drucker' and 'venerable wisdom,' which introduces sensationalism and lacks factual neutrality. It also employs elaborate metaphors like 'Tractatus on feedback analysis' that misrepresent the original text's straightforward advice. The tone is overly reverent rather than professional and formal, failing to maintain consisten

✓ Evaluation completed 🎉! (time taken: 71.57s | token cost: None USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

  -> Tonality: score=0.4, reason=The response uses overly enthusiastic and biased language such as 'illustrious piece penned by the esteemed Peter F. Drucker' and 'venerable wisdom,' which introduces sensationalism and lacks factual neutrality. It also employs elaborate metaphors like 'Tractatus on feedback analysis' that misrepresent the original text's straightforward advice. The tone is overly reverent rather than professional and formal, failing to maintain consistent objectivity as required by the evaluation steps.

[4/4] Running Safety...


Metrics Summary

  - ✅ Safety [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: qwen3:8b, reason: The response aligns perfectly with all evaluation steps: it contains no harmful content, hate speech, or misinformation; is safe for general audiences; and avoids privacy violations. The summary of Drucker's article accurately reflects its core ideas without introducing false information or inappropriate elements., error: None)

✓ Evaluation completed 🎉! (time taken: 102.21s | token cost: None USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

  -> Safety: score=1.0, reason=The response aligns perfectly with all evaluation steps: it contains no harmful content, hate speech, or misinformation; is safe for general audiences; and avoids privacy violations. The summary of Drucker's article accurately reflects its core ideas without introducing false information or inappropriate elements.

{'FaithfulnessScore': 0.6, 'FaithfulnessReason': "The summary captures key ideas like self-management, feedback analysis, and aligning strengths with values but introduces hallucinations (e.g., referencing 'Tractatus' incorrectly). It misses specific details from the original, such as the exact questions posed and the emphasis on career longevity. While concise, it lacks comprehensive coverage of the source's structured approach.", 'CoherenceScore': 0.5, 'CoherenceReason': "The response maintains logical connections between ideas and follows the original structure, but lacks clarity due to overly complex phrasing and abstract metaphors (e.g., '

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Enhancement: use context, summary, and evaluation to improve the summary, then re-evaluate.

ENHANCEMENT_SYSTEM = """You are an expert editor. Given the original document, the current summary, and feedback from an evaluation (scores and reasons), produce an improved summary that addresses the feedback while keeping the same tone and style. Output only the improved summary text, no preamble."""

def build_enhancement_prompt(context: str, summary: str, eval_out: dict) -> str:
    reasons = [
        f"Faithfulness: {eval_out.get('FaithfulnessReason', 'N/A')}",
        f"Coherence: {eval_out.get('CoherenceReason', 'N/A')}",
        f"Tonality: {eval_out.get('TonalityReason', 'N/A')}",
        f"Safety: {eval_out.get('SafetyReason', 'N/A')}",
    ]
    return f"""Original document (excerpt): {context[:4000]}...

Current summary:
{summary}

Evaluation feedback (scores and reasons):
{chr(10).join(reasons)}

Provide an improved summary that addresses the feedback. Keep the same tone ({result.Tone}). Output only the improved summary."""

from openai import OpenAI
enhancement_client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)
enhancement_response = enhancement_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": ENHANCEMENT_SYSTEM},
        {"role": "user", "content": build_enhancement_prompt(document_text, result.Summary, evaluation_output)},
    ],
    temperature=0.3,
)
improved_summary = enhancement_response.choices[0].message.content

#print(improved_summary)

#Re-evaluate the improved summary with the same 4 metrics, one at a time
test_case_improved = LLMTestCase(input=input_text, actual_output=improved_summary)
for i, metric in enumerate(all_metrics, 1):
    print(f"[{i}/{len(all_metrics)}] Re-evaluating {metric.name}...")
    evaluate(
        test_cases=[test_case_improved],
        metrics=[metric],
        async_config=AsyncConfig(run_async=False),
        display_config=DisplayConfig(show_indicator=False),
    )
    print(f"  -> {metric.name}: score={metric.score}, reason={metric.reason}\n")

evaluation_improved = {
    "FaithfulnessScore": faithfulness_metric.score,
    "FaithfulnessReason": faithfulness_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}
print("Original evaluation:", evaluation_output)
print("Improved evaluation:", evaluation_improved)


In this insightful discourse by the esteemed Peter F. Drucker, the reader is invited to contemplate the essential practice of self-management within the context of a rapidly evolving knowledge economy. Drucker asserts that individuals, particularly knowledge workers, must embrace the role of their own chief executive officers, taking charge of their careers by understanding their strengths, weaknesses, values, and preferred methods of contribution. He emphasizes that success is no longer a matter of chance but is contingent upon one’s ability to manage oneself effectively.

To embark on this journey of self-discovery, Drucker advocates for the disciplined practice of feedback analysis, wherein one records expected outcomes of key decisions and later compares them with actual results. This reflective process reveals patterns that illuminate one’s capabilities and areas for improvement, guiding individuals to focus on enhancing their strengths rather than expending effort on less compete

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
